In [1]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
    module="keras.src.export.tf2onnx_lib"
)
import tensorflow as tf
import pandas as pd
from src.constants import *
from src.helpers import load_img, build_dataset, decode_model_output, load_MY_model, get_encode_funs
import editdistance

In [2]:
df = pd.read_csv(TEST_TSV, sep='\t', header=None, names=['file', 'label'])
df = df.dropna(subset=['label']).reset_index(drop=True)
image_paths = [TEST_DIR + "/" + f for f in df["file"].values]
labels = df["label"].values

In [3]:
label_to_int, label_to_str = get_encode_funs()
model = load_MY_model()
test_ds = build_dataset(image_paths, labels, 32, label_to_int)
pred_y = model.predict(test_ds)

I0000 00:00:1769201443.699155    8968 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5797 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 SUPER, pci bus id: 0000:01:00.0, compute capability: 7.5


49/49 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step


In [ ]:
truths = []
preds = []
preds_probs = []
for imgs, labels in test_ds:
    pred = model(imgs, training=False)
    preds_probs.append(pred)
    preds.extend(decode_model_output(pred, label_to_str))
    for seq in labels.numpy():
        seq = seq[seq != 0]
        truths.append(b"".join(label_to_str(seq).numpy()).decode("utf-8"))

# Character error rate (average levenshtein distance, the lower the better)
total_edits = 0
total_chars = 0
for p, t in zip(preds, truths):
    total_edits += editdistance.eval(p, t)
    total_chars += len(t)
cer = total_edits / max(1, total_chars)

# Exact word match (the higher the better)
correct = sum(p == t for p, t in zip(preds, truths))
ewm = correct / max(1, len(truths))

# Blank ratio (how many characters are being predicted as blanks)
# (generally the lower the better)
logits = tf.concat(preds_probs, axis=0)
pred_ids = tf.argmax(logits, axis=-1)
blank_idx = logits.shape[-1] - 1
blanks = tf.equal(pred_ids, blank_idx)
blank_cnt = tf.reduce_sum(tf.cast(blanks, tf.float32))
total_steps = tf.cast(tf.size(pred_ids), tf.float32)
blank_ratio = (blank_cnt / tf.maximum(1., total_steps)).numpy()

# Length ratio (how long the predicted label is vs. the actual length)
# (1.0 is the ideal value)
pred_lengths = [len(p) for p in preds]
truth_lengths = [len(t) for t in truths]
len_ratio = sum(pred_lengths) / max(1, sum(truth_lengths))

print(f"Cer: {cer}") # not good, but somewhat workable
print(f"Ewm: {ewm}")
print(f"Blank ratio: {blank_ratio}")
print(f"Length ratio: {len_ratio}")

{'img_input': <tf.Tensor: shape=(32, 200, 800, 1), dtype=float32, numpy=
array([[[[1.        ],
         [1.        ],
         [1.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        [[1.        ],
         [1.        ],
         [1.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        [[0.9941177 ],
         [0.9941177 ],
         [0.99466914],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        ...,

        [[1.        ],
         [1.        ],
         [1.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        [[1.        ],
         [1.        ],
         [1.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.        ]],

        [[1.        ],
         [1.        ],
         [1.        ],
         ...,
         [0.        ],
         [0.        ],
         [0.  

KeyboardInterrupt: 